In [ ]:
import geopandas as gpd
import numpy as np
from shapely.geometry import Polygon

# Define input file paths (replace with your actual paths)
fields_path = ""  # Path to the field geometries file (shapefile/geojson)
substate_path = ""  # Path to the sub-state administrative boundaries file (shapefile/geojson)

# Load data
fields = gpd.read_file(fields_path)
substate = gpd.read_file(substate_path)

# Ensure both datasets use the same CRS
fields = fields.to_crs(substate.crs)

# Spatial join: Assign administrative boundaries to fields
fields_with_admin = gpd.sjoin(fields, substate, how="inner", predicate="intersects")

# Calculate metrics
def calculate_metrics(fields_gdf):
    # Field sizes (area)
    field_areas = fields_gdf.geometry.area
    avg_field_size = field_areas.mean()
    std_field_size = field_areas.std()
    var_field_size = field_areas.var()
    range_field_size = field_areas.max() - field_areas.min()
    
    # Number of vertices
    num_vertices = fields_gdf.geometry.apply(lambda geom: len(geom.exterior.coords) if isinstance(geom, Polygon) else 0)
    avg_vertices = num_vertices.mean()
    std_vertices = num_vertices.std()
    range_vertices = num_vertices.max() - num_vertices.min()

    # Field orientation (based on the longest side)
    def calculate_orientation(polygon):
        if not isinstance(polygon, Polygon):
            return np.nan
        max_line = max(polygon.exterior.coords[:-1], key=lambda coord: Polygon([coord]).length)
        dx = max_line[1][0] - max_line[0][0]
        dy = max_line[1][1] - max_line[0][1]
        angle = np.arctan2(dy, dx) * 180 / np.pi  # Convert to degrees
        return angle
    
    field_orientations = fields_gdf.geometry.apply(calculate_orientation)
    avg_field_orientation = field_orientations.mean()

    # Irregularity measure (compactness: perimeter^2 / area)
    compactness = fields_gdf.geometry.apply(
        lambda geom: geom.length**2 / geom.area if geom.area > 0 else np.nan
    )
    avg_irregularity = compactness.mean()

    # Output metrics as a dictionary
    metrics = {
        "avg_field_size": avg_field_size,
        "std_field_size": std_field_size,
        "var_field_size": var_field_size,
        "range_field_size": range_field_size,
        "avg_vertices": avg_vertices,
        "std_vertices": std_vertices,
        "range_vertices": range_vertices,
        "avg_field_orientation": avg_field_orientation,
        "avg_irregularity": avg_irregularity,
    }
    return metrics

# Calculate metrics for the joined fields
metrics = calculate_metrics(fields_with_admin)

# Print metrics
for key, value in metrics.items():
    print(f"{key}: {value}")
